# 08 — Figures 8 and 9: HMI vertical/horizontal field and PIL contours

Reproduces **Figures 8** (AR 13663) and **9** (AR 13664) of the paper. For each date we render a two-panel composite:

* **Left:** vertical magnetic field $B_z$ from the SHARP CEA `Br` segment.
* **Right:** horizontal field magnitude $B_h = \sqrt{B_t^2 + B_p^2}$ with Polarity Inversion Line contours overlaid at $\pm 300$ G (cyan = negative, lime = positive). The conventional PIL threshold $\pm 150$ G was replaced by $\pm 300$ G for these regions to minimise visual clutter in the highly sheared field — see Section 2.1 of the paper.

| AR | HARPNUM | Dates |
|---|---:|---|
| AR 13663 | 11142 | 2024-05-03, 2024-05-05, 2024-05-06 |
| AR 13664 | 11149 | 2024-05-08, 2024-05-09, 2024-05-10 |

> **Requirements.** This notebook expects the SHARP CEA FITS cubes to be available locally under `data/sharp/` (see notebook `02`). The expected filename pattern is `hmi.sharp_cea_720s.<HARPNUM>.<YYYYMMDD>_000000_TAI.{Br,Bt,Bp}.fits`.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from sunpy.map import Map

SHARP_DIR = Path('..').resolve() / 'data' / 'sharp'
FIG_DIR   = Path('..').resolve() / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
})


In [ ]:
def sharp_paths(harpnum, date_str):
    """Return the Br/Bt/Bp FITS paths for one HARPNUM and one date.

    `date_str` must be in 'YYYY-MM-DD' format."""
    yyyymmdd = date_str.replace('-', '')
    template = f'hmi.sharp_cea_720s.{harpnum}.{yyyymmdd}_000000_TAI.{{seg}}.fits'
    return tuple(SHARP_DIR / template.format(seg=seg) for seg in ('Br', 'Bt', 'Bp'))


def process_sharp_data(file_br, file_bt, file_bp):
    """Load the three SHARP CEA segments and return (Br map, Bh array)."""
    m_br = Map(file_br)
    m_bt = Map(file_bt)
    m_bp = Map(file_bp)
    bh = np.sqrt(m_bt.data ** 2 + m_bp.data ** 2)
    return m_br, bh


def plot_paper_figure(m_br, bh, title_str, outfile):
    """Two-panel figure (Bz left, Bh + PIL right) saved to `outfile`."""
    br = m_br.data
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    # --- Left: vertical field Bz ---
    im1 = axes[0].imshow(br, cmap='gray', origin='lower',
                         vmin=-1500, vmax=1500,
                         extent=[0, br.shape[1], 0, br.shape[0]])
    axes[0].set_title(f'Vertical Magnetic Field ($B_z$)\n{title_str}')
    axes[0].set_xlabel('Pixels (CEA)')
    axes[0].set_ylabel('Pixels (CEA)')
    cbar1 = plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    cbar1.set_label('Flux Density [Gauss]')

    # --- Right: horizontal field Bh + PIL contours ---
    im2 = axes[1].imshow(bh, cmap='magma', origin='lower',
                         vmin=0, vmax=2500,
                         extent=[0, bh.shape[1], 0, bh.shape[0]])
    axes[1].set_title(f'Horizontal Field Magnitude ($B_h$)\n{title_str}')
    axes[1].set_xlabel('Pixels (CEA)')

    br_smooth = gaussian_filter(br, sigma=5)
    axes[1].contour(br_smooth, levels=[-300, 300],
                    colors=['cyan', 'lime'], linewidths=1.5, alpha=0.9)

    axes[1].text(0.05, 0.95,
                 r'PIL Contours ($\pm 300$ G):' + '\nCyan(−) / Lime(+)',
                 transform=axes[1].transAxes, color='white', fontsize=10,
                 fontweight='bold', va='top',
                 bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', pad=3))

    cbar2 = plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    cbar2.set_label('Horizontal Magnitude [Gauss]')

    plt.tight_layout()
    plt.savefig(outfile, dpi=300)
    plt.show()


## Figure 8 — AR 13663 (HARPNUM 11142)


In [ ]:
for date in ['2024-05-03', '2024-05-05', '2024-05-06']:
    files = sharp_paths(11142, date)
    m_br, bh = process_sharp_data(*files)
    title = f'AR 13663 — {date} — 00:00 UTC'
    outfile = FIG_DIR / f"Figure_Magnetic_Morphology_{date.replace('-','_')[5:]}_13663.pdf"
    plot_paper_figure(m_br, bh, title, outfile)


## Figure 9 — AR 13664 (HARPNUM 11149)


In [ ]:
for date in ['2024-05-08', '2024-05-09', '2024-05-10']:
    files = sharp_paths(11149, date)
    m_br, bh = process_sharp_data(*files)
    title = f'AR 13664 — {date} — 00:00 UTC'
    outfile = FIG_DIR / f"Figure_Magnetic_Morphology_{date.replace('-','_')[5:]}_13664.pdf"
    plot_paper_figure(m_br, bh, title, outfile)
